# Four held-out customers

All four ids below fall on the held-out side. The rule is the sentence from lesson 09.

`9305-CDSKC` is a leave the rule flags. She is female, not a senior, with no partner and no dependents. Tenure is 8. She has a phone and multiple lines, fiber optic internet, no online security, no online backup, device protection, no tech support, and both streams. The contract is month-to-month, billing is paperless, and payment is an electronic check. Monthly charges are 99.65 and the total is 820.50. Fiber and the electronic check are both true, so the sentence flags her. The label is `Yes`.

`9959-WOFKT` has fiber optic internet and a monthly charge of 106.70, with tenure 71. The contract is two year, and the label is `No`. The sentence leaves him unflagged because the contract is not month-to-month.

`6067-NGCEU` is month-to-month fiber optic, tenure 65, monthly charge 111.05, paying by credit card. The sentence flags a leave. The label is `No`.

`3668-QPYBK` is month-to-month DSL, tenure 2, monthly charge 53.85, paying by mailed check. The sentence leaves him unflagged. The label is `Yes`.


In [1]:
# Locate the telco table beside this notebook, or under the course root.
import csv
from pathlib import Path

def data_path():
    here = Path.cwd()
    name = "Telco-Customer-Churn.csv"
    for folder in [here, *here.parents]:
        nested = folder / "16-Customer Churn" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists():
            return direct
    raise FileNotFoundError(name)

def read_customers():
    with data_path().open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))
# Four held-out rows: a hit, a long fiber stay, a false flag, and a miss.
import hashlib

rows = read_customers()

def held(customer_id):
    digest = hashlib.md5(customer_id.encode("utf-8")).hexdigest()
    return int(digest, 16) % 5 == 0

def leave_rule(row):
    return row["Contract"] == "Month-to-month" and (
        row["InternetService"] == "Fiber optic"
        or row["PaymentMethod"] == "Electronic check"
    )

by_id = {row["customerID"]: row for row in rows}
hit = by_id["9305-CDSKC"]
stay = by_id["9959-WOFKT"]
false_flag = by_id["6067-NGCEU"]
miss = by_id["3668-QPYBK"]
for row in (hit, stay, false_flag, miss):
    print(row["customerID"], held(row["customerID"]), leave_rule(row), row["Churn"])
    assert held(row["customerID"])
assert hit["tenure"] == "8" and hit["MonthlyCharges"] == "99.65"
assert hit["TotalCharges"] == "820.5" and hit["InternetService"] == "Fiber optic"
assert hit["PaymentMethod"] == "Electronic check" and hit["Churn"] == "Yes"
assert leave_rule(hit)
assert stay["Contract"] == "Two year" and stay["InternetService"] == "Fiber optic"
assert stay["tenure"] == "71" and stay["MonthlyCharges"] == "106.7" and stay["Churn"] == "No"
assert not leave_rule(stay)
assert false_flag["Contract"] == "Month-to-month"
assert false_flag["InternetService"] == "Fiber optic" and false_flag["tenure"] == "65"
assert false_flag["Churn"] == "No" and leave_rule(false_flag)
assert miss["Contract"] == "Month-to-month" and miss["InternetService"] == "DSL"
assert miss["PaymentMethod"] == "Mailed check" and miss["tenure"] == "2"
assert miss["MonthlyCharges"] == "53.85" and miss["Churn"] == "Yes"
assert not leave_rule(miss)


9305-CDSKC True True Yes
9959-WOFKT True False No
6067-NGCEU True True No
3668-QPYBK True False Yes


## Pitfall

`9959-WOFKT` would be flagged by a fiber-only sentence and is a stay on a two-year contract. `3668-QPYBK` leaves on month-to-month DSL with a mailed check, which the winning sentence does not flag. Both rows are inside the 1417, so they are part of the count $tp = 244$, $fp = 252$, $fn = 121$, $tn = 800$, one customer at a time.

## Summary

`9305-CDSKC` is a flagged leave. `9959-WOFKT` is an unflagged two-year fiber stay. `6067-NGCEU` is a flagged stay. `3668-QPYBK` is an unflagged leave.
